# Multimodal Clinical Discharge Risk

Binary prediction of **non-routine discharge** (`LAMA` / `Discharged on Request` / `Shifted to Higher Center`) from a clinical narrative plus tabular vitals and labs.

> Data: 5,000 de-identified hospital discharge records. Results are a research demonstration, not clinical evidence.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import transformers  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40"])

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "dataset.py").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Repo root not found - clone or upload the project first.")
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.train import resolve_device

DATA_PATH = REPO_ROOT / "data" / "hospital_anynimised.csv"
MODELS_DIR = REPO_ROOT / "models"
device = resolve_device("auto")

print("Repository root:", REPO_ROOT)
print("Device:", device)

## S1 : Data loading and preprocessing

`Age` and `Distance_From_Hospital_km` arrive as **text ranges** (`"<18"`, `"75+"`, `"5-15 km"`, `"50+ km"`). `parse_age` / `parse_distance` convert them to representative midpoints before imputation.

Earlier versions left these unparsed: every paediatric and 75+ patient was median-imputed, and distance was silently dropped, so the model trained on 34 of its advertised 35 features.

The dataset is **not committed** (see `.gitignore`); place `data/hospital_anynimised.csv` in `data/` before running.

In [ ]:
import numpy as np

from src.dataset import parse_age, parse_distance, prepare_datasets

print("Age parsing :", {value: parse_age(value) for value in ["<18", "18-29", "30-44", "75+"]})
print("Distance    :", {value: parse_distance(value) for value in ["0-5 km", "5-15 km", "50+ km"]})

prepared = prepare_datasets(DATA_PATH)

print()
print(f"Train rows: {len(prepared.y_train)} | Test rows: {len(prepared.y_test)}")
print(f"Tabular feature dimension: {prepared.X_train.shape[1]}")
print(f"Feature names: {len(prepared.feature_names)}")
print(f"Missing values: train={int(np.isnan(prepared.X_train).sum())}, test={int(np.isnan(prepared.X_test).sum())}")
print(f"Class balance (train): {np.bincount(prepared.y_train)}")
print(f"Positive rate: {prepared.y_train.mean():.3f}")

## S2 : Tokenization and DataLoader construction

Clinical text is truncated/padded to 128 tokens with the Bio_ClinicalBERT tokenizer. A stratified 15% of the training split is held out for validation — that split, never the test set, chooses the decision threshold.

In [ ]:
from src.dataset import make_loaders
from src.train import load_tokenizer

tokenizer, tokenizer_name = load_tokenizer("emilyalsentzer/Bio_ClinicalBERT")
splits = make_loaders(prepared, tokenizer)

batch = next(iter(splits.train_loader))

print("Tokenizer:", tokenizer_name)
print("Train / val / test :",
      len(splits.train_loader.dataset),
      len(splits.val_loader.dataset),
      len(splits.test_loader.dataset))
print("input_ids:", tuple(batch["input_ids"].shape),
      "| attention_mask:", tuple(batch["attention_mask"].shape))
print("tabular:", tuple(batch["tabular"].shape), "| label:", tuple(batch["label"].shape))
print("pos_weight:", round(float(splits.pos_weight), 4))

## S3 : Multimodal fusion architecture

Bio_ClinicalBERT `[CLS]` embedding (768-d) concatenated with a tabular MLP (35 → 128 → 64) gives an 832-d fused vector, classified by 128 → 32 → 1.

In [ ]:
import torch

from src.model import BioClinicalMultiModalFusion

model = BioClinicalMultiModalFusion(
    num_tabular_features=prepared.X_train.shape[1],
    bert_model_name=tokenizer_name,
).to(device)

model.eval()
with torch.no_grad():
    logits = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["tabular"].to(device),
    )

print("Parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))
print("Logits:", tuple(logits.shape))
print("Sample probabilities:", torch.sigmoid(logits[:5]).cpu().numpy().round(4))

## S4 : Two-stage training

Stage 1 freezes the text encoder for 3 warmup epochs (heads only); stage 2 fine-tunes everything for 5 epochs with a linear warmup schedule. The best checkpoint by validation ROC-AUC is restored.

Set `RUN_TRAINING = True` to retrain (~10-15 min on a GPU, hours on CPU). By default the shipped checkpoint is loaded. Model weights are **not committed to git** (~414 MB), so on a fresh clone either fetch `models/best_multimodal_model.pt` from Drive or set `RUN_TRAINING = True`.

In [ ]:
RUN_TRAINING = False

from src.train import TrainConfig, set_seed

set_seed(42)
config = TrainConfig()

if RUN_TRAINING:
    from src.train import train

    trained = train(prepared, tokenizer, tokenizer_name, config, device, MODELS_DIR)
    model = trained["model"]
    splits = trained["splits"]
    checkpoint = trained["checkpoint"]
else:
    checkpoint = torch.load(
        MODELS_DIR / "best_multimodal_model.pt", map_location=device, weights_only=False
    )
    model = BioClinicalMultiModalFusion(
        num_tabular_features=checkpoint["feature_dim"],
        bert_model_name=checkpoint["tokenizer_name"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("Loaded checkpoint:", checkpoint["stage_epoch"])
    print("Best validation ROC-AUC:", round(checkpoint["best_val_auc"], 4))

threshold = checkpoint["decision_threshold"]
print("Decision threshold (F1-optimal on validation):", round(threshold, 4))

## S5 : Test evaluation

Metrics are computed on the untouched 20% test split at the validation-chosen threshold. `evaluate_test` also writes `test_metrics.json` and `multimodal_evaluation_plots.png` to the repository root.

In [ ]:
import torch.nn as nn

from src.train import evaluate_test

criterion = nn.BCEWithLogitsLoss(pos_weight=splits.pos_weight.to(device))
metrics = evaluate_test(model, splits, criterion, threshold, device, REPO_ROOT)